In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size ='5' color='black'>ch15. 데이터베이스 연동</font></b>
# 1절. SQLite 데이터 베이스 연결

- SQLite 데이터베이스 : 
    - 데이터베이스 관리 시스템(DBMS) 서버 설치 없이, 간단한 파일 단위로 SQL을 이용해 데이터에 접근할 수 있는 디스크 기반 데이터베이스 
    - C 언어로 작성된 라이브러리
    - 프로토타입(시제품) 제작에 활용
- 프로젝트단계 : 분석 $\rightarrow$ 설계 $\rightarrow$ 구현 $\rightarrow$ 테스트 $\rightarrow$ 고객 배포 $\rightarrow$ 유지보수
    - 프로토타입 단계: SQLite (빠르고 가벼운 검증용 DB)
    - 완제품/실서버 단계: Oracle, MySQL, MariaDB, PostgreSQL, MSSQL, Amazon AuroraDB 등 (대용량·고성능 처리용 DB)

- [DB Browser for SQLite](https://sqlitebrowser.org/) $\rightarrow$ "DB Browser for SQLite - .zip (no installer) for 64-bit Windows" 다운로드후 압축 풀기

## 1.1 SQLite browser 및 sqlite3 pkg load

In [17]:
import sqlite3
sqlite3.sqlite_version

'3.40.1'

In [18]:
import pandas as pd
pd.__version__

'1.5.3'

## 1.2 데이터베이스 연결
1. DB 연결 객체 생성 (파일이 없으면 자동 생성, 있으면 연결)
2. 커서 객체 생성
3. 원하는 로직 수행 (SQL 전송 및 결과 처리)
4. 커서 객체 해제
5. DB 연결 객체 해제

In [23]:
# DB연결 (여기서 에러가 날 경우 VC_redist.x64.exe 설치)
conn = sqlite3.connect('data/ch15_example.db')
conn

In [24]:
# 커서 객체 생성 : 커서는 SQL문 실행시키고, 결과를 받는 객체
cursor = conn.cursor()
cursor

In [27]:
cursor.execute('''
    CREATE TABLE MEMBER (
        NAME TEXT,
        AGE  INT,
        EMAIL TEXT 
    )
''')

In [26]:
cursor.execute('DROP TABLE MEMBER')

In [28]:
# INSERT, UPDATE, DELETE문 실행결과 : cursor.rowcount (변경/삭제/삽입된 행의 개수를 정수로 반환)
sql = 'INSERT INTO MEMBER VALUES (\'마길동\', 25, \'h@h.com\')'
cursor.execute(sql) # sql 전송
print('insert, update, delete문의 수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신길동', 30, 's@h.com')"
cursor.execute(sql) 
print('수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신림동', 35, 'sil@h.com')"
cursor.execute(sql) 
print('수행 결과 행수 :', cursor.rowcount)

insert, update, delete문의 수행 결과 행수 : 1
수행 결과 행수 : 1
수행 결과 행수 : 1


In [29]:
conn.commit() # commit()과 rollback()은 데이터베이스의 상태를 직접적으로 변경하는 DML 실행 시에만 사용

In [30]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE") # SELECT sql문 전송

In [31]:
# SELECT문 실행결과를 받는 함수들
    # fetchone() : 결과에서 한 행만 가져올 때 (데이터가 없으면 None), 튜플 ( )
    # fetchall() : 조회된 모든 행을 한 번에 가져올 때, 튜플 리스트 [( ), ( )]
    # fetchmany(n) : 지정한 n개의 행만 가져올 때, 튜플 리스트 [( ), ( )]
cursor.fetchall()

[('마길동', 25, 'h@h.com'), ('신길동', 30, 's@h.com'), ('신림동', 35, 'sil@h.com')]

In [32]:
cursor.fetchall() # 한 번 읽어서 소모한 데이터는 커서 내에서 다시 접근하거나 재사용할 수 없음

[]

In [33]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchall()
members

[('마길동', 25, 'h@h.com'), ('신길동', 30, 's@h.com'), ('신림동', 35, 'sil@h.com')]

In [34]:
# 한 줄씩 가져와 dict list에 append하기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    member = cursor.fetchone() # 결과에서 한 행만 가져오기
    if member is None:
        break
    members.append({'name':member[0], 'age':member[1], 'email':member[2]})
members

[{'name': '마길동', 'age': 25, 'email': 'h@h.com'},
 {'name': '신길동', 'age': 30, 'email': 's@h.com'},
 {'name': '신림동', 'age': 35, 'email': 'sil@h.com'}]

In [35]:
class Member:
    'Member 테이블의 내용을 받은 객체 타입'
    def __init__(self, name, age, email):
        self.name = name
        self.age  = age
        self.email = email
    def __str__(self):
        return "{}\t{}\t{}".format(self.name, self.age, self.email)
m = Member('홍길동', 25, 'h@h.com')
print(m)

홍길동	25	h@h.com


In [36]:
dbmember = ('홍길동', 25, 'h@h.com')
m = Member(*dbmember)
print(m)

홍길동	25	h@h.com


In [37]:
# 한 줄씩 가져와 객체 list에 append
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    dbmember = cursor.fetchone()
    if dbmember is None:
        break
    member = Member(*dbmember)
    members.append(member)
for mem in members:
    print(mem)

마길동	25	h@h.com
신길동	30	s@h.com
신림동	35	sil@h.com


In [38]:
# 최상위 n행 읽어오기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchmany(2)
members

[('마길동', 25, 'h@h.com'), ('신길동', 30, 's@h.com')]

In [39]:
cursor.close()
conn.close()

## 1.3 SQL구문에 파라미터 사용하기
- qmark(DB에 따라 불가한 경우가 있음)
- named(추천)

In [40]:
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN ('홍길동','신길동')")
cursor.fetchall()

[('신길동', 30, 's@h.com')]

In [41]:
# 파라미터 사용하기 : qmark 방법 이용
name1 = input('검색할 이름1 :')
name2 = input('검색할 이름2 :')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN ('{name1}', '{name2}')")
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN (?, ?)", (name1, name2))
cursor.fetchall()

검색할 이름1 :신길동
검색할 이름2 :신림동


[('신길동', 30, 's@h.com'), ('신림동', 35, 'sil@h.com')]

In [42]:
# 파라미터 사용하기 : named 방법 이용
name1 = input('검색할 이름1 :')
name2 = input('검색할 이름2 :')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN ('{name1}', '{name2}')")
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN (:name1, :name2)", {'name1':name1,
                                                                      'name2':name2})
cursor.fetchall()

검색할 이름1 :마마마
검색할 이름2 :가가가


[]

In [27]:
# 파라미터 이용하기 : named 방법
name = input('이름 : ')
try:
    age = int(input('나이(숫자) : '))
except:
    print('유효하지 않은 나이를 입력할 경우 초기화 됩니다')
    age = 1
email = input('이메일 : ')
cursor.execute("INSERT INTO MEMBER VALUES (:name, :age, :email)", 
              {'name':name, 'age':age, 'email':email}) # sql 전송
conn.commit()
print('수행 결과 행 수 :', cursor.rowcount)

이름 : 김철수
나이(숫자) : 24
이메일 : serigh@gov.co.kr
수행 결과 행 수 : 1


In [43]:
cursor.close()
conn.close()

# 2절. 오라클 데이터베이스 연결
- pip install cx_oracle(~ 11g), pip install oracledb(12 ~)

In [60]:
import cx_Oracle
cx_Oracle.__version__

'8.3.0'

In [61]:
# conn 얻어오는 방법 1 : makedsn 사용

# 1. DSN(Data Source Name) 정보 생성
oracle_dsn = cx_Oracle.makedsn(host="localhost", port=1521, sid='xe')

# 2. DB 연결 (위치 기반 인자 전달: ID, PW, DSN)
# conn = cx_Oracle.connect(user='scott', password='tiger', dsn=oracle_dsn)
conn = cx_Oracle.connect('scott', 'tiger', oracle_dsn)

# 3. DB 연결 해제
conn.close()

In [62]:
# conn 얻어오는 방법 2 : Easy Connect 문자열 사용 (에러 발생할 경우 VC_redist.x64.exe 설치)

#conn = cx_Oracle.connect(user='scott', password='tiger', dsn='localhost:1521/xe')
conn = cx_Oracle.connect('scott', 'tiger', 'localhost:1521/xe')
conn

<cx_Oracle.Connection to scott@localhost:1521/xe>

In [63]:
# cursor 객체 생성하고 sql문 전송 & 결과 받기
cursor = conn.cursor()
sql = "SELECT EMPNO NO, ENAME, JOB, MGR, HIREDATE, SAL, COMM, DEPTNO FROM EMP"
cursor.execute(sql)
emps = cursor.fetchall()

In [64]:
for emp in emps:
    print(emp)

(7369, 'SMITH', 'CLERK', 7902, datetime.datetime(1980, 12, 17, 0, 0), 800.0, None, 20)
(7499, 'ALLEN', 'SALESMAN', 7698, datetime.datetime(1981, 2, 20, 0, 0), 1600.0, 300.0, 30)
(7521, 'WARD', 'SALESMAN', 7698, datetime.datetime(1981, 2, 22, 0, 0), 1250.0, 500.0, 30)
(7566, 'JONES', 'MANAGER', 7839, datetime.datetime(1981, 4, 2, 0, 0), 2975.0, None, 20)
(7654, 'MARTIN', 'SALESMAN', 7698, datetime.datetime(1981, 9, 28, 0, 0), 1250.0, 1400.0, 30)
(7698, 'BLAKE', 'MANAGER', 7839, datetime.datetime(1981, 5, 1, 0, 0), 2850.0, None, 30)
(7782, 'CLARK', 'MANAGER', 7839, datetime.datetime(1981, 6, 9, 0, 0), 2450.0, None, 10)
(7788, 'SCOTT', 'ANALYST', 7566, datetime.datetime(1982, 12, 9, 0, 0), 3000.0, None, 20)
(7839, 'KING', 'PRESIDENT', None, datetime.datetime(1981, 11, 17, 0, 0), 5000.0, None, 10)
(7844, 'TURNER', 'SALESMAN', 7698, datetime.datetime(1981, 9, 8, 0, 0), 1500.0, 0.0, 30)
(7876, 'ADAMS', 'CLERK', 7788, datetime.datetime(1983, 1, 12, 0, 0), 1100.0, None, 20)
(7900, 'JAMES', 'CL

In [65]:
cursor.description

[('NO', <cx_Oracle.DbType DB_TYPE_NUMBER>, 5, None, 4, 0, 0),
 ('ENAME', <cx_Oracle.DbType DB_TYPE_VARCHAR>, 10, 10, None, None, 1),
 ('JOB', <cx_Oracle.DbType DB_TYPE_VARCHAR>, 9, 9, None, None, 1),
 ('MGR', <cx_Oracle.DbType DB_TYPE_NUMBER>, 5, None, 4, 0, 1),
 ('HIREDATE', <cx_Oracle.DbType DB_TYPE_DATE>, 23, None, None, None, 1),
 ('SAL', <cx_Oracle.DbType DB_TYPE_NUMBER>, 11, None, 7, 2, 1),
 ('COMM', <cx_Oracle.DbType DB_TYPE_NUMBER>, 11, None, 7, 2, 1),
 ('DEPTNO', <cx_Oracle.DbType DB_TYPE_NUMBER>, 3, None, 2, 0, 1)]

In [66]:
# 컬럼명 나열
[descipt[0] for descipt in cursor.description]

['NO', 'ENAME', 'JOB', 'MGR', 'HIREDATE', 'SAL', 'COMM', 'DEPTNO']

In [67]:
import pandas as pd
emp_df = pd.DataFrame(emps, 
                     columns=[descipt[0] for descipt in cursor.description])
emp_df

,NO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7369,SMITH,CLERK,7902.0,1980-12-17,800.0,NaN,20
1,7499,ALLEN,SALESMAN,7698.0,1981-02-20,1600.0,300.0,30
2,7521,WARD,SALESMAN,7698.0,1981-02-22,1250.0,500.0,30
3,7566,JONES,MANAGER,7839.0,1981-04-02,2975.0,NaN,20
4,7654,MARTIN,SALESMAN,7698.0,1981-09-28,1250.0,1400.0,30
5,7698,BLAKE,MANAGER,7839.0,1981-05-01,2850.0,NaN,30
6,7782,CLARK,MANAGER,7839.0,1981-06-09,2450.0,NaN,10
7,7788,SCOTT,ANALYST,7566.0,1982-12-09,3000.0,NaN,20
8,7839,KING,PRESIDENT,NaN,1981-11-17,5000.0,NaN,10
9,7844,TURNER,SALESMAN,7698.0,1981-09-08,1500.0,0.0,30


In [68]:
# 사용자로부터 검색할 이름을 받아 해당 데이터 출력

sql = "SELECT * FROM EMP WHERE ENAME=(:ename)"

ename = input('검색할 이름 : ').upper()
cursor.execute(sql, {'ename':ename})

emp = cursor.fetchone() # 결과가 있으면 해당 데이터를 튜플로, 결과가 없으면 None으로

if emp:
    columns = [descript[0] for descript in cursor.description]
    df = pd.DataFrame([emp], columns=columns)
    display(df)
else:
    print('해당 이름이 없습니다')

검색할 이름 : scott


,EMPNO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7788,SCOTT,ANALYST,7566,1982-12-09,3000.0,None,20


In [69]:
for col, data in zip(columns, emp):
    print("{} : {}".format(col, data if data is not None else '-'))

EMPNO : 7788
ENAME : SCOTT
JOB : ANALYST
MGR : 7566
HIREDATE : 1982-12-09 00:00:00
SAL : 3000.0
COMM : -
DEPTNO : 20


In [70]:
cursor.close()
conn.close()

# 3절. MySQL 연결

| 라이브러리 | 특징 |
|:--|:--|
| **mysql-connector-python** | MySQL 공식 커넥터, python으로 구현 |
| **pyMySQL** | 커뮤니티에서 만든 서드파트 라이브러리(경량), python으로 구현, 널리 쓰임 |

- pip install pyMySQL

In [43]:
%pip install pyMySQL

     ---------------------------------------- 45.7/45.7 kB 2.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pymysql

# 1. DB 연결 객체 생성
conn = pymysql.connect(
    host='127.0.0.1',       # DB 서버 주소
    # port=3306,            # 포트 번호 (기본값 3306 생략 가능)
    user='root',            # 계정명
    password='mysql',       # 비밀번호
    database='devdb',       # 사용할 데이터베이스
    charset='utf8mb4',      # 한글 및 이모지 깨짐 방지
    autocommit=True         # DML 자동 커밋 활성화
)

# 2. 커서 객체 생성
cursor = conn.cursor()

# 3. SQL 실행 및 데이터 가져오기
sql = 'SELECT * FROM person'
cursor.execute(sql)         # SQL 전송 및 결과 처리 준비
person = cursor.fetchall()  # 모든 조회 결과 받아오기 (튜플 리스트 형태)

print(person)

# 4. 자원 해제 (생성의 역순)
cursor.close()
conn.close()

((1001, 'bill', 'president', None, datetime.date(1989, 1, 10), Decimal('7000'), None, 10), (1111, 'smith', 'manager', 1001, datetime.date(1990, 12, 17), Decimal('1000'), None, 10), (1112, 'ally', 'salesman', 1116, datetime.date(1991, 2, 20), Decimal('1600'), Decimal('500'), 30), (1113, 'word', 'salesman', 1116, datetime.date(1992, 2, 24), Decimal('1450'), Decimal('300'), 30), (1114, 'james', 'manager', 1001, datetime.date(1990, 4, 12), Decimal('3975'), None, 20), (1116, 'johnson', 'manager', 1001, datetime.date(1991, 5, 1), Decimal('3550'), None, 30), (1118, 'martin', 'analyst', 1111, datetime.date(1991, 9, 9), Decimal('3450'), None, 10), (1121, 'kim', 'clerk', 1114, datetime.date(1990, 12, 8), Decimal('4000'), None, 20), (1123, 'lee', 'salesman', 1116, datetime.date(1991, 9, 23), Decimal('1200'), Decimal('0'), 30), (1226, 'park', 'analyst', 1111, datetime.date(1990, 1, 3), Decimal('2500'), None, 10))


In [5]:
import pandas as pd
import numpy as np
df = pd.DataFrame(person, 
                  columns=[descript[0] for descript in cursor.description])
#df.fillna(np.nan)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pno       10 non-null     int64  
 1   pname     10 non-null     object 
 2   job       10 non-null     object 
 3   manager   9 non-null      float64
 4   hiredate  10 non-null     object 
 5   sal       10 non-null     object 
 6   comm      3 non-null      object 
 7   dno       10 non-null     int64  
dtypes: float64(1), int64(2), object(5)
memory usage: 768.0+ bytes


In [6]:
df.isna().sum()

pno         0
pname       0
job         0
manager     1
hiredate    0
sal         0
comm        7
dno         0
dtype: int64

In [7]:
df['hiredate'] = df['hiredate'].astype('datetime64[ns]')

In [8]:
df = df.astype({'sal':'float64', 'comm':np.float64})
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   pno       10 non-null     int64         
 1   pname     10 non-null     object        
 2   job       10 non-null     object        
 3   manager   9 non-null      float64       
 4   hiredate  10 non-null     datetime64[ns]
 5   sal       10 non-null     float64       
 6   comm      3 non-null      float64       
 7   dno       10 non-null     int64         
dtypes: datetime64[ns](1), float64(3), int64(2), object(2)
memory usage: 768.0+ bytes


# 4절. 연습문제

## oracle연동
- 회원가입 | 전체조회 | 이름찾기 | 메일삭제 | csv내보내기 | 종료

### 0. 처음실행

In [71]:
def load_conn():
    global conn # 전역변수로 사용함을 선언
    import cx_Oracle
    conn = cx_Oracle.connect("scott", "tiger", "localhost:1521/xe")
load_conn()

### 1. 회원입력

In [109]:
def fn1_insert_member():
    '사용자로부터 이름, 전화, 이메일, 나이, 등급(1~5)을 입력받아 DB에 insert 한다'
    
    # 사용자로부터 데이터 입력 받기
    name = input('이름 : ')
    phone = input('전화 : ')
    email = input('메일 : ')
    try:
        age = int(input('나이 : '))
        if age<0 :
            age = 0
    except:
        print('유효하지 않은 나이 입력 시 0으로 초기화')
        age = 0
    try:
        grade = int(input('등급(1~5) : '))
        if grade < 1:
            grade = 1
        elif grade > 5:
            grade = 5
    except:
        print('유효하지 않은 등급 입력 시 1로 초기화')
        grade = 1
    
    # SQL문 전송 및 결과 받기
    cursor = conn.cursor()
    sql = "INSERT INTO MEMBER VALUES (:name, :phone, :email, :age, :grade)"
    cursor.execute(sql, {'name':name, 'phone':phone, 'email':email, 'age':age, 'grade':grade})
    
    if cursor.rowcount:
        conn.commit()
        print(name + '님 회원가입 완료')
        
    cursor.close()
    
# fn1_insert_member() 

### 2. 전체조회

In [110]:
def fn2_display_members():
    'member 테이블의 내용을 데이터프레임으로 display'
    
    import pandas as pd
    
    cursor = conn.cursor()
    
    sql = """SELECT NAME, PHONE, EMAIL, AGE, GRADE 
                FROM MEMBER 
                ORDER BY AGE"""
    cursor.execute(sql)
    
    members = cursor.fetchall() # 데이터가 있으면 튜플 리스트, 데이터가 없으면 빈 리스트
    
    if members:
        columns = [descript[0] for descript in cursor.description]
        df = pd.DataFrame(members, columns=columns)
        display(df)
    else:
        print('입력된 회원이 없습니다')
        
    cursor.close()
    
# fn2_display_members()

### 3. 이름으로 조회

In [111]:
def fn3_search_name():
    '찾고자하는 이름을 입력받아 해당 이름의 데이터를 데이터프레임으로 display'
    
    name1 = input('검색할 이름 : ')
    
    import pandas as pd
    
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM MEMBER WHERE NAME IN :name", {'name': name1})
    members = cursor.fetchall()
    
    if members:
        columns = [descript[0] for descript in cursor.description]
        df = pd.DataFrame(members, columns=columns)
        display(df)
    else:
        print('입력된 회원이 없습니다')
        
    cursor.close()
    
# fn3_search_name()

### 4. 메일로 삭제

In [112]:
def fn4_delete_member():
    '삭제하고자 하는 회원의 메일을 입력받아 회원정보 delete 하기'
    
    email1 = input('삭제할 회원의 메일 : ')
    
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM MEMBER WHERE UPPER(EMAIL)=UPPER(:email)", {'email': email1})
    members = cursor.fetchall()
    
    if members:
        names = [member[0] for member in members]
        print(f'메일이 {email1}인 {names[0]} 회원 외 {len(members)-1}명의 정보를 삭제하겠습니다')
        cursor.execute("DELETE FROM MEMBER WHERE UPPER(EMAIL)=UPPER(:email)", {'email': email1})
        conn.commit()
    else:
        print('데이터베이스에 없는 메일입니다')
        
    cursor.close()

# fn4_delete_member()

### 5. csv 내보내기
- data/ch15_member.csv로 회원정보 내보내기

In [113]:
def fn5_save_csv():
    import pandas as pd
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM MEMBER")
    members = cursor.fetchall()
    
    if members:
        columns = [descript[0] for descript in cursor.description]
        df = pd.DataFrame(members, columns=columns)
        df.to_csv('data/ch15_member.csv', index=False)
        print('csv 백업 완료')
    else:
        print('입력된 회원이 없습니다')
        
    cursor.close()

# fn5_save_csv()

### 6. 기능들 합하기

In [117]:
def main():
    while True:
        menu = input("1. 회원 입력 | 2. 전체 조회 | 3. 이름 찾기 | 4. 정보 삭제 | 5. CSV 백업 | 9. 종료")
        if menu=='1':
            fn1_insert_member()
        elif menu=='2':
            fn2_display_members()
        elif menu=='3':
            fn3_search_name()
        elif menu=='4':
            fn4_delete_member()
        elif menu=='5':
            fn5_save_csv()
        elif menu=='9':
            conn.close()
            break
        else:
            print('잘못된 번호 입니다.')

if __name__=='__main__':
    global conn
    import cx_Oracle
    conn = cx_Oracle.connect("scott","tiger","localhost:1521/xe")
#     import sqlite3
#     conn = sqlite3.connect('data/ch15_member.db')

In [118]:
main()

1. 회원 입력 | 2. 전체 조회 | 3. 이름 찾기 | 4. 정보 삭제 | 5. CSV 백업 | 9. 종료2


,NAME,PHONE,EMAIL,AGE,GRADE
0,김길동,010-4567-8912,h@h.com,24,3
1,홍길동,010-9999-9999,H@h.com,25,1
2,김길동,010-8888-8888,H@h.com,25,1
3,홍길동,010-1234-5678,h@h.com,33,4


1. 회원 입력 | 2. 전체 조회 | 3. 이름 찾기 | 4. 정보 삭제 | 5. CSV 백업 | 9. 종료4
삭제할 회원의 메일 : h@h.com
메일이 h@h.com인 김길동 회원 외 3명의 정보를 삭제하겠습니다
1. 회원 입력 | 2. 전체 조회 | 3. 이름 찾기 | 4. 정보 삭제 | 5. CSV 백업 | 9. 종료9
